# Лабораторна робота 3 — Градієнтний спуск для множинної регресії

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn-регресія **не дозволена** для базових завдань.

## Налаштування

In [1]:
!mamba install numpy pandas matplotlib scikit-learn


mambajs 0.21.1

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.841 seconds
  Name                Version    Build                Channel
---------------------------------------------------------------------------
+ brotli-python       1.2.0      py313ha26e73d_2      emscripten-forge-4x
+ certifi             2026.5.20  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.4.7      pyhd8ed1ab_0         conda-forge
+ idna                3.17       pyhcf101f3_0         conda-forge
+ joblib              1.5.3      py313h1804a44_3      emscripten-forge-4x
+ pandas              3.0.3      np23py313h1e705a5_0  emscripten-forge-4x
+ pysocks             1.7.1      py313h1804a44_3      emscripten-forge-4x
+ python-tzdata       2026.2     pyhd8ed1ab_0         conda-forge
+ requests            2.34.2     pyhcf101f3_0         conda-forge
+ scikit-learn      

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Градієнт RSS відносно ваги *wᵢ*:
```
gradient_i = 2 · dot(errors, feature_i)     де  errors = Xw − y
```
Зупинка відбувається, коли ‖gradient‖₂ < `tolerance`.

## Завдання 1 — Підготовка даних та `get_numpy_data()`

Завантажте `kc_house_data.csv`. Розбийте **20 % навчання / 80 % тест** (`random_state=0`).

Реалізуйте `get_numpy_data(dataframe, features, output)`, яка:
1. Додає стовпець з одиницями ліворуч від матриці ознак (для вільного члена).
2. Повертає `(feature_matrix, output_array)` як масиви NumPy.

In [3]:
sales = pd.read_csv('kc_house_data.csv')
train_data, test_data = train_test_split(sales, test_size=0.8, random_state=0)


In [4]:
def get_numpy_data(dataframe, features, output):
    # Додаємо стовпець константи 1 (вільний член / intercept)
    dataframe = dataframe.copy()
    dataframe['constant'] = 1.0

    # Формуємо матрицю ознак: спочатку стовпець 'constant', потім решта ознак
    all_features = ['constant'] + list(features)
    feature_matrix = dataframe[all_features].to_numpy(dtype=float)

    output_array = dataframe[output].to_numpy(dtype=float)
    return feature_matrix, output_array


### Перевірка

In [5]:
example_features, example_output = get_numpy_data(sales, ['sqft_living'], 'price')
print('Перший рядок матриці ознак:', example_features[0, :])  # має бути [1.0, <sqft>]
print('Перше значення виходу:    ', example_output[0])


Перший рядок матриці ознак: [1.00e+00 1.18e+03]
Перше значення виходу:     221900.0


## Завдання 2 — Реалізація `predict_output()`

Використовуйте **один виклик `np.dot`** — без явних циклів.

In [6]:
def predict_output(feature_matrix, weights):
    # Один виклик np.dot — без явних циклів
    return np.dot(feature_matrix, weights)


### Перевірка — перші два передбачення мають бути ≈1181 і ≈2571

In [7]:
my_weights = np.array([1.0, 1.0])
test_preds = predict_output(example_features, my_weights)
print(f'передбачення[0]: {test_preds[0]:.1f}  (очікується ≈1181)')
print(f'передбачення[1]: {test_preds[1]:.1f}  (очікується ≈2571)')


передбачення[0]: 1181.0  (очікується ≈1181)
передбачення[1]: 2571.0  (очікується ≈2571)


## Завдання 3 — Реалізація `feature_derivative()`

Похідна RSS відносно однієї ваги — це `2 · dot(errors, feature)`. Реалізуйте як один вираз NumPy.

In [8]:
def feature_derivative(errors, feature):
    # gradient_i = 2 * dot(errors, feature_i)
    return 2 * np.dot(errors, feature)


### Перевірка — похідна відносно константної ознаки має дорівнювати `-2 · sum(prices)`

In [9]:
zero_weights = np.array([0.0, 0.0])
zero_preds   = predict_output(example_features, zero_weights)
errors       = zero_preds - example_output              # = -prices when weights=0

deriv_constant = feature_derivative(errors, example_features[:, 0])
expected       = -2 * np.sum(example_output)
print(f'Обчислено: {deriv_constant:.2e}')
print(f'Очікується: {expected:.2e}')
print(f'Збіг: {np.isclose(deriv_constant, expected)}')


Обчислено: -2.33e+10
Очікується: -2.33e+10
Збіг: True


## Завдання 4 — Градієнтний спуск

Реалізуйте `regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance)`. Функція оновлює всі ваги одночасно на кожній ітерації та повертає їх, коли ‖gradient‖₂ < `tolerance`.

Запустіть з такими параметрами на **навчальних** даних:
```
features        = ['sqft_living']
initial_weights = [-47000., 1.]
step_size       = 7e-12
tolerance       = 2.5e7
```
Вкажіть навчені ваги. Яка передбачувана ціна першого будинку в **тестовій** вибірці? Обчисліть RSS на всій тестовій вибірці.

In [10]:
def regression_gradient_descent(feature_matrix, output,
                                initial_weights, step_size, tolerance):
    weights   = np.array(initial_weights, dtype=float)
    converged = False

    while not converged:
        # 1. Обчислюємо передбачення
        predictions = predict_output(feature_matrix, weights)

        # 2. Обчислюємо похибки
        errors = predictions - output

        gradient_sum_squares = 0.0
        # Зберігаємо нові ваги окремо, щоб оновлення було одночасним
        new_weights = np.copy(weights)

        for i in range(len(weights)):
            # 3. Похідна RSS відносно ваги i
            derivative = feature_derivative(errors, feature_matrix[:, i])

            # 4. Накопичуємо суму квадратів градієнтів
            gradient_sum_squares += derivative ** 2

            # 5. Оновлюємо вагу
            new_weights[i] = weights[i] - step_size * derivative

        weights = new_weights

        # 6. Перевірка збіжності
        gradient_magnitude = sqrt(gradient_sum_squares)
        if gradient_magnitude < tolerance:
            converged = True

    return weights


### Запуск Моделі 1 — одна ознака

In [11]:
simple_feature_matrix, output = get_numpy_data(train_data, ['sqft_living'], 'price')

simple_weights = regression_gradient_descent(
    simple_feature_matrix, output,
    initial_weights=[-47000., 1.],
    step_size=7e-12,
    tolerance=2.5e7
)
print('Навчені ваги:', simple_weights)
print(f'  intercept (w0): {simple_weights[0]:.4f}')
print(f'  sqft_living (w1): {simple_weights[1]:.4f}')


Навчені ваги: [-46999.88018846    280.51005013]
  intercept (w0): -46999.8802
  sqft_living (w1): 280.5101


### Оцінка Моделі 1 на тестовій вибірці

In [12]:
test_feature_matrix, test_output = get_numpy_data(test_data, ['sqft_living'], 'price')

# Передбачена ціна першого будинку в тестовій вибірці
test_predictions = predict_output(test_feature_matrix, simple_weights)
print(f'Передбачена ціна (будинок 0): ${test_predictions[0]:,.2f}')
print(f'Реальна ціна    (будинок 0): ${test_output[0]:,.2f}')

# RSS на всій тестовій вибірці
rss_model1 = np.sum((test_predictions - test_output) ** 2)
print(f'\nТестова RSS Моделі 1: {rss_model1:.6e}')


Передбачена ціна (будинок 0): $354,129.49
Реальна ціна    (будинок 0): $297,000.00

Тестова RSS Моделі 1: 1.203845e+15


## Бонус — Додайте другу ознаку

Повторіть градієнтний спуск з `['sqft_living', 'sqft_living15']` та параметрами:
```
initial_weights = [-100000., 1., 1.]
step_size       = 4e-12
tolerance       = 1e9
```
Обчисліть RSS на тестовій вибірці та порівняйте з моделлю з однією ознакою. Порівняйте передбачену ціну першого тестового будинку з обох моделей з реальною ціною.

In [13]:
# Бонус — модель з двома ознаками
bonus_feature_matrix, output_train2 = get_numpy_data(
    train_data, ['sqft_living', 'sqft_living15'], 'price')

bonus_weights = regression_gradient_descent(
    bonus_feature_matrix, output_train2,
    initial_weights=[-100000., 1., 1.],
    step_size=4e-12,
    tolerance=1e9
)
print('Навчені ваги (Модель 2):', bonus_weights)
print(f'  intercept (w0):       {bonus_weights[0]:.4f}')
print(f'  sqft_living (w1):     {bonus_weights[1]:.4f}')
print(f'  sqft_living15 (w2):   {bonus_weights[2]:.4f}')

# Оцінка на тестовій вибірці
test_bonus_matrix, _ = get_numpy_data(
    test_data, ['sqft_living', 'sqft_living15'], 'price')
bonus_predictions = predict_output(test_bonus_matrix, bonus_weights)

rss_model2 = np.sum((bonus_predictions - test_output) ** 2)

print('\n--- Порівняння моделей ---')
print(f'Реальна ціна (будинок 0):          ${test_output[0]:>14,.2f}')
print(f'Модель 1 – передбачення (будинок 0): ${test_predictions[0]:>14,.2f}')
print(f'Модель 2 – передбачення (будинок 0): ${bonus_predictions[0]:>14,.2f}')
print()
print(f'Тестова RSS Моделі 1: {rss_model1:.6e}')
print(f'Тестова RSS Моделі 2: {rss_model2:.6e}')
if rss_model2 < rss_model1:
    improvement = (rss_model1 - rss_model2) / rss_model1 * 100
    print(f'\n Модель 2 краща: RSS зменшився на {improvement:.2f}%')
else:
    print('\nМодель 1 не поступається Моделі 2')


Навчені ваги (Модель 2): [-9.99998826e+04  2.43610628e+02  6.54855462e+01]
  intercept (w0):       -99999.8826
  sqft_living (w1):     243.6106
  sqft_living15 (w2):   65.4855

--- Порівняння моделей ---
Реальна ціна (будинок 0):          $    297,000.00
Модель 1 – передбачення (будинок 0): $    354,129.49
Модель 2 – передбачення (будинок 0): $    342,007.65

Тестова RSS Моделі 1: 1.203845e+15
Тестова RSS Моделі 2: 1.186047e+15

 Модель 2 краща: RSS зменшився на 1.48%
